<a href="https://colab.research.google.com/github/yweslakarep123/Krispy2/blob/main/project_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
  import subprocess, os

  # 1. EGL config so libglvnd can find the Nvidia EGL driver
  NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
  os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
  if not os.path.exists(NVIDIA_ICD):
      with open(NVIDIA_ICD, 'w') as f:
          f.write('{\n  "file_format_version":"1.0.0",\n'
                  '  "ICD":{"library_path":"libEGL_nvidia.so.0"}\n}\n')

  # 2. Set render backend BEFORE any mujoco / dm_control import
  os.environ['MUJOCO_GL']         = 'egl'
  os.environ['PYOPENGL_PLATFORM'] = 'egl'

  # 3. Install missing packages (h5py, gymnasium-robotics, minari)
  #    dm_control, torch, numpy, tqdm are already in Colab.
  subprocess.run([
      'pip', 'install', '-q',
      'dm_control>=1.0.38',          # MuJoCo Python bindings
      'gymnasium-robotics>=1.2.4',   # FrankaKitchen-v1 env
      'h5py',                        # read .hdf5 dataset files
      'minari',                      # fallback dataset source (d4rl successor)
  ], check=True)

  print("Setup complete.")


Setup complete.


In [2]:
"""
FlowPolicy + FQL Refinement + RST Data Augmentation
for Franka Kitchen (kitchen-complete-v0 dataset)

Compatible with Google Colab (Python 3.12, GPU runtime).
Does NOT require d4rl — dataset is downloaded directly as HDF5.

────────────────────────────────────────────────────────────────
HOW TO RUN ON COLAB
────────────────────────────────────────────────────────────────
Paste and run SECTION 0 in its own cell first, then run the rest.
"""

# =============================================================================
# SECTION 0  --  COLAB SETUP  (run this cell alone, then continue)
# =============================================================================
# +-------------------------------------------------------------------------+
# |  Copy everything inside this block into a Colab cell and run it.        |
# |  A kernel restart is NOT needed after installation.                     |
# +-------------------------------------------------------------------------+
#


# =============================================================================
# SECTION 1  --  IMPORTS
# =============================================================================
import os, sys, math, copy, random, warnings, urllib.request
warnings.filterwarnings("ignore")

# Must be set before any MuJoCo / dm_control import
os.environ.setdefault("MUJOCO_GL",         "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from typing import Dict, List, Tuple, Optional
from tqdm import tqdm

# MuJoCo via dm_control  (direct `import mujoco` crashes on Colab)
try:
    from dm_control import mujoco as _dm_mujoco   # noqa: F401
    print("dm_control loaded")
except ImportError:
    print("WARNING: dm_control not found -- run Section 0 first")

# gymnasium-robotics  (registers FrankaKitchen-v1)
try:
    import gymnasium as gym
    import gymnasium_robotics                      # noqa: F401
    gym.register_envs(gymnasium_robotics)
    print(f"gymnasium {gym.__version__} + gymnasium-robotics loaded")
except ImportError:
    print("WARNING: gymnasium-robotics not found -- run Section 0 first")

try:
    import h5py
    print("h5py loaded")
except ImportError:
    print("WARNING: h5py not found -- run Section 0 first")

try:
    import minari
    HAS_MINARI = True
    print("minari loaded")
except ImportError:
    HAS_MINARI = False



# =============================================================================
# SECTION 2  --  CONFIGURATION
# =============================================================================
class Config:
    # Environment
    ENV_NAME        = "FrankaKitchen-v1"
    ENV_TASKS       = ["microwave", "kettle", "light switch", "slide cabinet"]
    STATE_DIM       = None          # resolved dynamically from the env
    ACTION_DIM      = 9             # 7 arm joints + 2 gripper
    MAX_EPISODE_LEN = 280

    # Dataset (downloaded from RAIL if not cached, or via Minari)
    DATASET_URLS = [
        "http://rail.eecs.berkeley.edu/datasets/offline_rl"
        "/kitchen/mini_kitchen_microwave_kettle_light_slider-v0.hdf5",
    ]
    DATASET_PATH = "./kitchen_complete-v0.hdf5"
    MINARI_DATASET_ID = "kitchen-complete-v1"

    # RST Data Augmentation
    RST_K       = 5
    RST_SEG_LEN = 40
    RST_N_AUG   = 200

    # Flow Policy (Rectified Flow)
    FLOW_HIDDEN    = 512
    FLOW_DEPTH     = 6
    FLOW_N_STEPS   = 10
    FLOW_T_EMB_DIM = 128

    # FQL
    FQL_HIDDEN = 512
    FQL_DEPTH  = 4
    FQL_GAMMA  = 0.99
    FQL_TAU    = 0.005
    FQL_ALPHA  = 8.0    # stronger distillation to flow prior
    FQL_BETA   = 0.2    # use flow action in TD target (stability blend)
    FQL_Q_CLIP = 150.0  # clamp target Q range
    FQL_ACTOR_DELAY = 2 # update actor every N critic steps

    REWARD_SCALE = 0.05 # reward scaling for critic stability

    # Training
    BATCH_SIZE  = 256
    LR_FLOW     = 3e-4
    LR_FQL      = 3e-4
    FLOW_EPOCHS = 300
    FQL_EPOCHS  = 200
    EVAL_EVERY  = 20
    N_EVAL_EPS  = 5

    # Misc
    SEED          = 42
    DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
    SAVE_PATH     = "./checkpoints"

cfg = Config()
os.makedirs(cfg.SAVE_PATH, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)
print(f"Device: {cfg.DEVICE}")


# =============================================================================
# SECTION 3  --  ENVIRONMENT WRAPPER
# =============================================================================
class FlatObsWrapper(gym.Wrapper):
    """
    Flatten dict/tuple/array observations into a single float32 vector.

    FrankaKitchen-v1 returns a Dict observation space where some sub-spaces
    report shape=None (e.g. achieved_goal / desired_goal).  Inspecting
    .spaces is therefore unreliable; instead we do a real reset() and
    measure the flattened array directly.
    """

    def _flatten(self, obs):
        """Recursively extract all leaf numeric values and concatenate."""
        def _collect(o):
            if isinstance(o, dict):
                parts = []
                for v in o.values():
                    parts.extend(_collect(v))
                return parts
            try:
                arr = np.asarray(o, dtype=np.float32).ravel()
                return [arr] if arr.size > 0 else []
            except (TypeError, ValueError):
                return []
        parts = _collect(obs)
        return np.concatenate(parts) if parts else np.zeros(1, dtype=np.float32)

    def __init__(self, env):
        super().__init__(env)

        # Probe the actual flat dim with a real reset (avoids None-shape issues)
        result   = env.reset()
        raw_obs  = result[0] if isinstance(result, tuple) else result
        probe    = self._flatten(raw_obs)
        flat_dim = probe.shape[0]

        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=(flat_dim,), dtype=np.float32)
        self._flat_dim = flat_dim
        print(f"[FlatObsWrapper] flat obs dim = {flat_dim}")

    def reset(self, **kwargs):
        result = self.env.reset(**kwargs)
        obs, info = result if isinstance(result, tuple) else (result, {})
        return self._flatten(obs), info

    def step(self, action):
        result = self.env.step(action)
        if len(result) == 5:                  # gymnasium API
            obs, rew, term, trunc, info = result
            done = bool(term or trunc)
        else:                                  # gym API
            obs, rew, done, info = result
        return self._flatten(obs), float(rew), done, info


def build_env():
    env = gym.make(
        cfg.ENV_NAME,
        tasks_to_complete=cfg.ENV_TASKS,
        terminate_on_tasks_completed=False,
        remove_task_when_completed=False,
    )
    return FlatObsWrapper(env)


def infer_state_dim() -> int:
    """Create a temporary env to read the flat obs dim."""
    env = build_env()
    dim = env.observation_space.shape[0]
    env.close()
    cfg.STATE_DIM = dim
    print(f"STATE_DIM = {dim}")
    return dim


# =============================================================================
# SECTION 4  --  DATASET LOADING  (no d4rl required)
# =============================================================================
def download_dataset():
    """Download the HDF5 dataset if not already cached.  Tries multiple URLs."""
    if os.path.exists(cfg.DATASET_PATH):
        size_mb = os.path.getsize(cfg.DATASET_PATH) / 1e6
        print(f"Found cached dataset: {cfg.DATASET_PATH}  ({size_mb:.1f} MB)")
        return True

    def _progress(n_blocks, block_size, total_size):
        done = min(n_blocks * block_size, total_size)
        pct  = done / total_size * 100 if total_size > 0 else 0
        print(f"\r  {done/1e6:.1f}/{total_size/1e6:.1f} MB  ({pct:.0f}%)",
              end="", flush=True)

    for url in cfg.DATASET_URLS:
        try:
            print(f"Downloading dataset from {url} ...")
            urllib.request.urlretrieve(url, cfg.DATASET_PATH, _progress)
            print(f"\nSaved to {cfg.DATASET_PATH}")
            return True
        except Exception as e:
            print(f"\n  Failed: {e}")
            if os.path.exists(cfg.DATASET_PATH):
                os.remove(cfg.DATASET_PATH)

    print("All direct download URLs failed.")
    return False


def _load_via_minari() -> Dict:
    """Load kitchen-complete dataset via Minari (official d4rl successor)."""
    dataset_id = cfg.MINARI_DATASET_ID
    print(f"[Minari] Loading dataset '{dataset_id}' ...")

    local_datasets = minari.list_local_datasets()
    if dataset_id not in local_datasets:
        print(f"[Minari] Downloading '{dataset_id}' ...")
        minari.download_dataset(dataset_id)

    dataset = minari.load_dataset(dataset_id)

    obs_all, acts_all, rews_all, dones_all = [], [], [], []
    for ep in dataset.iterate_episodes():
        obs  = np.asarray(ep.observations, dtype=np.float32)
        acts = np.asarray(ep.actions, dtype=np.float32)
        rews = np.asarray(ep.rewards, dtype=np.float32)

        if isinstance(obs[0], dict):
            flat = []
            for o in obs:
                parts = [np.asarray(v, dtype=np.float32).ravel()
                         for v in o.values()]
                flat.append(np.concatenate(parts))
            obs = np.array(flat)

        T = min(len(obs), len(acts))
        obs, acts, rews = obs[:T], acts[:T], rews[:T]
        terms = np.zeros(T, dtype=bool)
        terms[-1] = True
        obs_all.append(obs)
        acts_all.append(acts)
        rews_all.append(rews)
        dones_all.append(terms)

    obs   = np.concatenate(obs_all).astype(np.float32)
    acts  = np.concatenate(acts_all).astype(np.float32)
    rews  = np.concatenate(rews_all).astype(np.float32)
    dones = np.concatenate(dones_all)

    print(f"[Minari] Loaded {len(obs)} transitions, "
          f"{dones.sum()} episode ends, obs_dim={obs.shape[1]}")
    return obs, acts, rews, dones


def load_kitchen_dataset() -> Dict:
    hdf5_ok = download_dataset()

    if hdf5_ok:
        with h5py.File(cfg.DATASET_PATH, "r") as f:
            obs      = f["observations"][:].astype(np.float32)
            acts     = f["actions"][:].astype(np.float32)
            rewards  = f["rewards"][:].astype(np.float32)
            terms    = f["terminals"][:].astype(bool)
            timeouts = (f["timeouts"][:].astype(bool)
                        if "timeouts" in f else np.zeros(len(obs), bool))
            dones    = terms | timeouts
        print(f"Loaded {len(obs)} transitions, {dones.sum()} episodes ends, "
              f"obs_dim={obs.shape[1]}")
    elif HAS_MINARI:
        print("Falling back to Minari ...")
        obs, acts, rewards, dones = _load_via_minari()
    else:
        raise RuntimeError(
            "Cannot download dataset: all URLs failed and minari is not "
            "installed.  Run:  pip install minari")

    if cfg.STATE_DIM is None:
        cfg.STATE_DIM = obs.shape[1]
        print(f"STATE_DIM inferred from dataset: {cfg.STATE_DIM}")

    D = obs.shape[1]
    if D > cfg.STATE_DIM:
        obs = obs[:, :cfg.STATE_DIM]
    elif D < cfg.STATE_DIM:
        obs = np.concatenate([obs,
                               np.zeros((len(obs), cfg.STATE_DIM - D),
                                        dtype=np.float32)], axis=1)

    obs_mean = obs.mean(0, keepdims=True)
    obs_std  = obs.std(0,  keepdims=True) + 1e-6
    obs_n    = (obs - obs_mean) / obs_std

    act_min = acts.min(0, keepdims=True)
    act_max = acts.max(0, keepdims=True)
    acts_n  = 2.0 * (acts - act_min) / (act_max - act_min + 1e-6) - 1.0

    rewards = rewards * cfg.REWARD_SCALE

    episodes, t0 = [], 0
    for t in range(len(obs_n)):
        if dones[t] or t == len(obs_n) - 1:
            ep = {"obs":  obs_n[t0:t+1],
                  "acts": acts_n[t0:t+1],
                  "rews": rewards[t0:t+1]}
            if len(ep["obs"]) > 1:
                episodes.append(ep)
            t0 = t + 1

    avg_len = np.mean([len(e["obs"]) for e in episodes]) if episodes else 0
    print(f"Parsed {len(episodes)} episodes  (avg len {avg_len:.1f})")

    return {"episodes": episodes,
            "obs_mean": obs_mean, "obs_std": obs_std,
            "act_min":  act_min,  "act_max": act_max}


# =============================================================================
# SECTION 5  --  RST  (Retrieval-based Sub-Trajectory Augmentation)
# =============================================================================
class RST:
    """
    Augments a small offline dataset by stitching sub-trajectories
    from different episodes via nearest-neighbour retrieval on states.

    For each synthetic trajectory:
      1. Pick a seed state from a random anchor episode.
      2. Retrieve K nearest-neighbour (ep, step) pairs by L2 state distance
         (excluding the anchor episode to ensure novelty).
      3. Append the matching sub-trajectory segment; update the current state.
      4. Repeat until MAX_EPISODE_LEN is reached.
    """

    def __init__(self, episodes: List[Dict], cfg: Config):
        self.eps = episodes
        self.cfg = cfg
        self._build_index()

    def _build_index(self):
        states, refs = [], []
        for eid, ep in enumerate(self.eps):
            for t in range(len(ep["obs"])):
                states.append(ep["obs"][t])
                refs.append((eid, t))
        self.index = np.array(states, dtype=np.float32)   # (N, D)
        self.refs  = refs
        print(f"[RST] Indexed {len(states)} states from {len(self.eps)} episodes")

    def _knn(self, query: np.ndarray, k: int, excl: int = -1):
        dists = ((self.index - query[None]) ** 2).sum(-1)
        for i, (eid, _) in enumerate(self.refs):
            if eid == excl:
                dists[i] = np.inf
        finite = int((dists < np.inf).sum())
        if finite == 0:
            return []
        kk   = min(k, finite)
        top  = np.argpartition(dists, kk - 1)[:kk]
        top  = top[np.argsort(dists[top])]
        return [self.refs[i] for i in top]

    def _stitch_one(self) -> Optional[Dict]:
        seg = self.cfg.RST_SEG_LEN
        K   = self.cfg.RST_K
        aeid = random.randrange(len(self.eps))
        aep  = self.eps[aeid]
        if len(aep["obs"]) < seg:
            return None
        t0    = random.randrange(max(1, len(aep["obs"]) - seg))
        state = aep["obs"][t0].copy()
        os_l, ac_l, rw_l = [], [], []
        for _ in range(self.cfg.MAX_EPISODE_LEN // seg):
            nbrs = self._knn(state, K, excl=aeid)
            if not nbrs:
                break
            eid, t = random.choice(nbrs)
            ep     = self.eps[eid]
            te     = min(t + seg, len(ep["obs"]))
            if te <= t:
                break
            os_l.append(ep["obs"][t:te])
            ac_l.append(ep["acts"][t:te])
            rw_l.append(ep["rews"][t:te])
            state = ep["obs"][te - 1].copy()
            if sum(len(x) for x in os_l) >= self.cfg.MAX_EPISODE_LEN:
                break
        if not os_l:
            return None
        return {"obs":  np.concatenate(os_l),
                "acts": np.concatenate(ac_l),
                "rews": np.concatenate(rw_l)}

    def augment(self, n_aug: int) -> List[Dict]:
        out, tries = [], 0
        pbar = tqdm(total=n_aug, desc="[RST] Augmenting")
        while len(out) < n_aug and tries < n_aug * 15:
            ep = self._stitch_one()
            if ep is not None:
                out.append(ep)
                pbar.update(1)
            tries += 1
        pbar.close()
        total = sum(len(e["obs"]) for e in out)
        print(f"[RST] {len(out)} augmented episodes ({total} steps)")
        return out


# =============================================================================
# SECTION 6  --  PYTORCH DATASET
# =============================================================================
class TransitionDataset(Dataset):
    def __init__(self, episodes: List[Dict]):
        os_l, ac_l, rw_l, no_l, dn_l = [], [], [], [], []
        for ep in episodes:
            T = len(ep["obs"])
            for t in range(T - 1):
                os_l.append(ep["obs"][t]);    ac_l.append(ep["acts"][t])
                rw_l.append(ep["rews"][t]);   no_l.append(ep["obs"][t+1])
                dn_l.append(0.0)
            if T > 0:
                os_l.append(ep["obs"][-1]);   ac_l.append(ep["acts"][-1])
                rw_l.append(ep["rews"][-1]);  no_l.append(ep["obs"][-1])
                dn_l.append(1.0)
        mk = lambda lst, t=torch.float32: torch.tensor(np.array(lst), dtype=t)
        self.obs      = mk(os_l)
        self.acts     = mk(ac_l)
        self.rewards  = mk(rw_l)
        self.next_obs = mk(no_l)
        self.dones    = mk(dn_l)
        print(f"TransitionDataset: {len(self.obs)} transitions")

    def __len__(self):  return len(self.obs)
    def __getitem__(self, i):
        return (self.obs[i], self.acts[i], self.rewards[i],
                self.next_obs[i], self.dones[i])


# =============================================================================
# SECTION 7  --  BUILDING BLOCKS
# =============================================================================
def sinusoidal_emb(t: torch.Tensor, dim: int) -> torch.Tensor:
    half  = dim // 2
    freqs = torch.exp(-math.log(10000) *
                      torch.arange(half, device=t.device, dtype=torch.float32)
                      / max(half - 1, 1))
    args  = t.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class MLP(nn.Module):
    def __init__(self, in_d, out_d, hid, depth, ln=True):
        super().__init__()
        layers, d = [], in_d
        for _ in range(depth - 1):
            layers += [nn.Linear(d, hid)]
            if ln: layers += [nn.LayerNorm(hid)]
            layers += [nn.SiLU()]
            d = hid
        layers += [nn.Linear(d, out_d)]
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


# =============================================================================
# SECTION 8  --  FLOW POLICY  (Rectified Flow)
# =============================================================================
class VelocityNet(nn.Module):
    """
    Velocity field v_theta(s, x_t, t) for rectified flow.

    Training:  L = || v_theta(s, x_t, t) - (a - z) ||^2
               x_t = (1-t)*z + t*a,   z ~ N(0,I),   t ~ U[0,1]

    Inference: Euler ODE  x <- x + v * dt  from t=0 to t=1
    """
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        H, T          = cfg.FLOW_HIDDEN, cfg.FLOW_T_EMB_DIM
        self.t_dim    = T
        self.inp_proj = nn.Linear(state_dim + action_dim, H)
        self.t_proj   = nn.Linear(T, H)
        self.blocks   = nn.ModuleList([
            nn.Sequential(nn.Linear(H, H), nn.LayerNorm(H), nn.SiLU())
            for _ in range(cfg.FLOW_DEPTH - 1)
        ])
        self.out      = nn.Linear(H, action_dim)

    def forward(self, s, x_t, t):
        h = self.inp_proj(torch.cat([s, x_t], -1))
        h = h + self.t_proj(sinusoidal_emb(t, self.t_dim))
        for blk in self.blocks:
            h = h + blk(h)
        return self.out(h)


class FlowPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        self.vel   = VelocityNet(state_dim, action_dim, cfg)
        self.adim  = action_dim
        self.steps = cfg.FLOW_N_STEPS

    def flow_loss(self, s, a):
        z   = torch.randn_like(a)
        t   = torch.rand(s.shape[0], device=s.device)
        x_t = (1 - t[:, None]) * z + t[:, None] * a
        return F.mse_loss(self.vel(s, x_t, t), a - z)

    @torch.no_grad()
    def sample(self, s, n_steps=None):
        n  = n_steps or self.steps
        x  = torch.randn(s.shape[0], self.adim, device=s.device)
        dt = 1.0 / n
        for i in range(n):
            t = torch.full((s.shape[0],), i * dt, device=s.device)
            x = x + self.vel(s, x, t) * dt
        return x.clamp(-1., 1.)

    def forward(self, s): return self.sample(s)


# =============================================================================
# SECTION 9  --  FQL  (Flow Q-Learning Refinement)
# =============================================================================
class TwinQ(nn.Module):
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        d = state_dim + action_dim
        self.q1 = MLP(d, 1, cfg.FQL_HIDDEN, cfg.FQL_DEPTH)
        self.q2 = MLP(d, 1, cfg.FQL_HIDDEN, cfg.FQL_DEPTH)

    def both(self, s, a):
        x = torch.cat([s, a], -1)
        return self.q1(x).squeeze(-1), self.q2(x).squeeze(-1)

    def min(self, s, a):
        q1, q2 = self.both(s, a)
        return torch.min(q1, q2)


class OnestepActor(nn.Module):
    """f_phi(s, z)  -- one-step actor distilled from the flow policy."""
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        self.net  = MLP(state_dim + action_dim, action_dim,
                        cfg.FQL_HIDDEN, cfg.FQL_DEPTH)
        self.adim = action_dim

    def forward(self, s, z=None):
        if z is None:
            z = torch.randn(s.shape[0], self.adim, device=s.device)
        return self.net(torch.cat([s, z], -1)).tanh()


class FQL(nn.Module):
    """
    Flow Q-Learning (FQL) offline RL refinement of a pre-trained FlowPolicy.

    Critic update:  standard Bellman TD with twin-Q soft target.
    Actor  update:  L = -Q(s, f_phi(s,z))  +  alpha * ||f_phi(s,z) - a_flow||^2
      - First term:  Q-gradient pushes actor towards high-value regions.
      - Second term: distillation keeps actor close to the safe flow prior.
    """
    def __init__(self, state_dim, action_dim, flow: FlowPolicy, cfg):
        super().__init__()
        self.flow   = flow
        self.actor  = OnestepActor(state_dim, action_dim, cfg)
        self.critic = TwinQ(state_dim, action_dim, cfg)
        self.target = copy.deepcopy(self.critic)
        for p in self.target.parameters():
            p.requires_grad_(False)
        self.gamma = cfg.FQL_GAMMA
        self.tau   = cfg.FQL_TAU
        self.alpha = cfg.FQL_ALPHA
        self.beta  = cfg.FQL_BETA
        self.q_clip = cfg.FQL_Q_CLIP
        self.adim  = action_dim

    @torch.no_grad()
    def soft_update(self):
        for p, pt in zip(self.critic.parameters(), self.target.parameters()):
            pt.data.lerp_(p.data, self.tau)

    def critic_loss(self, s, a, r, s2, d):
        with torch.no_grad():
            z2 = torch.randn(s.shape[0], self.adim, device=s.device)
            a_actor = self.actor(s2, z2)
            a_flow  = self.flow.sample(s2)
            a2 = self.beta * a_flow + (1.0 - self.beta) * a_actor
            q_t = r + self.gamma * (1 - d) * self.target.min(s2, a2)
            q_t = q_t.clamp(-self.q_clip, self.q_clip)
        q1, q2 = self.critic.both(s, a)
        return F.smooth_l1_loss(q1, q_t) + F.smooth_l1_loss(q2, q_t)

    def actor_loss(self, s):
        z    = torch.randn(s.shape[0], self.adim, device=s.device)
        a1   = self.actor(s, z)
        q    = self.critic.min(s, a1)
        with torch.no_grad():
            af = self.flow.sample(s)         # flow teacher (frozen)
        dist = F.mse_loss(a1, af)
        loss = -q.mean() + self.alpha * dist
        return loss, q.mean().detach(), dist.detach()

    @torch.no_grad()
    def act(self, s): return self.actor(s)


# =============================================================================
# SECTION 10  --  EVALUATION
# =============================================================================
def run_eval(env, policy_fn, obs_mean, obs_std,
             act_min, act_max, n_eps, device) -> Dict:
    rewards, tasks = [], []
    for _ in range(n_eps):
        result = env.reset()
        obs    = result[0] if isinstance(result, tuple) else result
        done, ep_r, ep_t = False, 0.0, set()
        while not done:
            # Normalise + align obs dim
            obs_n = (obs - obs_mean.squeeze()) / obs_std.squeeze()
            if len(obs_n) < cfg.STATE_DIM:
                obs_n = np.pad(obs_n, (0, cfg.STATE_DIM - len(obs_n)))
            else:
                obs_n = obs_n[:cfg.STATE_DIM]
            s_t  = torch.tensor(obs_n, dtype=torch.float32,
                                device=device).unsqueeze(0)
            a_t  = policy_fn(s_t).cpu().numpy()[0]
            # De-normalise action
            a_np = (a_t + 1.0) / 2.0 * (act_max - act_min).squeeze() \
                   + act_min.squeeze()
            a_np = np.clip(a_np, env.action_space.low, env.action_space.high)
            obs, r, done, info = env.step(a_np)
            ep_r += r
            for tk in info.get("completed_tasks", []):
                ep_t.add(tk)
        rewards.append(ep_r)
        tasks.append(len(ep_t))
    return {"mean_reward": float(np.mean(rewards)),
            "std_reward":  float(np.std(rewards)),
            "mean_tasks":  float(np.mean(tasks))}


# =============================================================================
# SECTION 11  --  TRAINING
# =============================================================================
def train():
    print("=" * 60)
    print("  FlowPolicy + FQL + RST  |  Franka Kitchen")
    print("=" * 60)

    # Resolve STATE_DIM from the live environment
    infer_state_dim()
    S, A = cfg.STATE_DIM, cfg.ACTION_DIM

    # Load + augment dataset
    data    = load_kitchen_dataset()
    eps_raw = data["episodes"]
    rst     = RST(eps_raw, cfg)
    eps_aug = rst.augment(cfg.RST_N_AUG)
    all_eps = eps_raw + eps_aug
    print(f"Data: {len(eps_raw)} original + {len(eps_aug)} RST = {len(all_eps)} eps")

    # DataLoader
    dset   = TransitionDataset(all_eps)
    loader = DataLoader(dset, batch_size=cfg.BATCH_SIZE,
                        shuffle=True, drop_last=True,
                        num_workers=2, pin_memory=(cfg.DEVICE == "cuda"))

    # Models
    flow = FlowPolicy(S, A, cfg).to(cfg.DEVICE)
    fql  = FQL(S, A, flow, cfg).to(cfg.DEVICE)

    flow_opt = Adam(flow.parameters(),       lr=cfg.LR_FLOW)
    q_opt    = Adam(fql.critic.parameters(), lr=cfg.LR_FQL)
    a_opt    = Adam(fql.actor.parameters(),  lr=cfg.LR_FQL)

    flow_sch = CosineAnnealingLR(flow_opt, cfg.FLOW_EPOCHS)
    q_sch    = CosineAnnealingLR(q_opt,    cfg.FQL_EPOCHS)
    a_sch    = CosineAnnealingLR(a_opt,    cfg.FQL_EPOCHS)

    eval_env = build_env()
    eval_env.reset(seed=cfg.SEED + 100)

    # ==========================================================================
    # PHASE 1 -- FlowPolicy pre-training (Rectified Flow BC)
    # ==========================================================================
    print("\n[Phase 1] Pre-training FlowPolicy ...")
    best_r1 = -np.inf

    for epoch in range(1, cfg.FLOW_EPOCHS + 1):
        flow.train()
        losses = []
        for s, a, *_ in loader:
            s, a = s.to(cfg.DEVICE), a.to(cfg.DEVICE)
            loss = flow.flow_loss(s, a)
            flow_opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(flow.parameters(), 1.0)
            flow_opt.step()
            losses.append(loss.item())
        flow_sch.step()

        if epoch % cfg.EVAL_EVERY == 0 or epoch == 1:
            flow.eval()
            m = run_eval(eval_env, flow,
                         data["obs_mean"], data["obs_std"],
                         data["act_min"],  data["act_max"],
                         cfg.N_EVAL_EPS,   cfg.DEVICE)
            print(f"  [Flow {epoch:4d}]  loss={np.mean(losses):.4f}  "
                  f"R={m['mean_reward']:.1f}+-{m['std_reward']:.1f}  "
                  f"tasks={m['mean_tasks']:.2f}")
            if m["mean_reward"] > best_r1:
                best_r1 = m["mean_reward"]
                torch.save(flow.state_dict(),
                           f"{cfg.SAVE_PATH}/flow_best.pt")

    print(f"[Phase 1] Best reward = {best_r1:.1f}")
    flow.load_state_dict(torch.load(f"{cfg.SAVE_PATH}/flow_best.pt",
                                    map_location=cfg.DEVICE))

    # ==========================================================================
    # PHASE 2 -- FQL Refinement
    # ==========================================================================
    print("\n[Phase 2] FQL Refinement ...")
    for p in flow.vel.parameters():         # freeze flow teacher
        p.requires_grad_(False)

    best_r2 = -np.inf

    global_step = 0
    for epoch in range(1, cfg.FQL_EPOCHS + 1):
        fql.train()
        q_ls, a_ls, q_vs = [], [], []

        for s, a, r, s2, d in loader:
            global_step += 1
            s, a  = s.to(cfg.DEVICE), a.to(cfg.DEVICE)
            r, s2 = r.to(cfg.DEVICE), s2.to(cfg.DEVICE)
            d     = d.to(cfg.DEVICE)

            # Critic
            cl = fql.critic_loss(s, a, r, s2, d)
            q_opt.zero_grad()
            cl.backward()
            nn.utils.clip_grad_norm_(fql.critic.parameters(), 1.0)
            q_opt.step()
            q_ls.append(cl.item())

            # Actor (delayed update for stability)
            if global_step % cfg.FQL_ACTOR_DELAY == 0:
                al, qv, _ = fql.actor_loss(s)
                a_opt.zero_grad()
                al.backward()
                nn.utils.clip_grad_norm_(fql.actor.parameters(), 1.0)
                a_opt.step()
                a_ls.append(al.item())
                q_vs.append(qv.item())

                fql.soft_update()

        q_sch.step()
        a_sch.step()

        if epoch % cfg.EVAL_EVERY == 0 or epoch == 1:
            fql.eval()
            m = run_eval(eval_env, fql.act,
                         data["obs_mean"], data["obs_std"],
                         data["act_min"],  data["act_max"],
                         cfg.N_EVAL_EPS,   cfg.DEVICE)
            print(f"  [FQL  {epoch:4d}]  "
                  f"q_loss={np.mean(q_ls):.4f}  "
                  f"a_loss={np.mean(a_ls) if a_ls else 0.0:.4f}  "
                  f"Q={np.mean(q_vs) if q_vs else 0.0:.2f}  "
                  f"R={m['mean_reward']:.1f}+-{m['std_reward']:.1f}  "
                  f"tasks={m['mean_tasks']:.2f}")
            if m["mean_reward"] > best_r2:
                best_r2 = m["mean_reward"]
                torch.save({"flow":   flow.state_dict(),
                            "actor":  fql.actor.state_dict(),
                            "critic": fql.critic.state_dict()},
                           f"{cfg.SAVE_PATH}/fql_best.pt")

    print(f"\n[Phase 2] Best FQL reward = {best_r2:.1f}")
    eval_env.close()
    return flow, fql, data


# =============================================================================
# SECTION 12  --  INFERENCE
# =============================================================================
def load_and_eval(ckpt_path: str, n_eps: int = 5):
    """Load best checkpoint and run evaluation rollouts."""
    infer_state_dim()
    ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE)
    flow = FlowPolicy(cfg.STATE_DIM, cfg.ACTION_DIM, cfg).to(cfg.DEVICE)
    fql  = FQL(cfg.STATE_DIM, cfg.ACTION_DIM, flow, cfg).to(cfg.DEVICE)
    flow.load_state_dict(ckpt["flow"])
    fql.actor.load_state_dict(ckpt["actor"])
    fql.eval()
    data = load_kitchen_dataset()
    env  = build_env()
    m    = run_eval(env, fql.act,
                    data["obs_mean"], data["obs_std"],
                    data["act_min"],  data["act_max"],
                    n_eps, cfg.DEVICE)
    env.close()
    print(f"[Eval] R={m['mean_reward']:.2f}+-{m['std_reward']:.2f}  "
          f"tasks={m['mean_tasks']:.2f}")
    return m


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    flow, fql, data = train()
    print("Checkpoints saved to:", cfg.SAVE_PATH)


dm_control loaded
gymnasium 1.2.3 + gymnasium-robotics loaded
h5py loaded
minari loaded
Device: cuda
  FlowPolicy + FQL + RST  |  Franka Kitchen
[FlatObsWrapper] flat obs dim = 81
STATE_DIM = 81
Found cached dataset: ./kitchen_complete-v0.hdf5  (0.6 MB)
Loaded 3680 transitions, 19 episodes ends, obs_dim=60
Parsed 19 episodes  (avg len 193.7)
[RST] Indexed 3680 states from 19 episodes


[RST] Augmenting: 100%|██████████| 200/200 [00:03<00:00, 54.51it/s]


[RST] 200 augmented episodes (26734 steps)
Data: 19 original + 200 RST = 219 eps
TransitionDataset: 30414 transitions
[FlatObsWrapper] flat obs dim = 81

[Phase 1] Pre-training FlowPolicy ...
  [Flow    1]  loss=0.5540  R=0.0+-0.0  tasks=0.00
  [Flow   20]  loss=0.1388  R=0.0+-0.0  tasks=0.00
  [Flow   40]  loss=0.1063  R=0.0+-0.0  tasks=0.00
  [Flow   60]  loss=0.0861  R=0.0+-0.0  tasks=0.00
  [Flow   80]  loss=0.0749  R=0.0+-0.0  tasks=0.00
  [Flow  100]  loss=0.0643  R=0.0+-0.0  tasks=0.00
  [Flow  120]  loss=0.0561  R=0.0+-0.0  tasks=0.00
  [Flow  140]  loss=0.0486  R=0.0+-0.0  tasks=0.00
  [Flow  160]  loss=0.0442  R=0.0+-0.0  tasks=0.00
  [Flow  180]  loss=0.0375  R=0.0+-0.0  tasks=0.00
  [Flow  200]  loss=0.0318  R=0.0+-0.0  tasks=0.00
  [Flow  220]  loss=0.0280  R=0.0+-0.0  tasks=0.00
  [Flow  240]  loss=0.0236  R=0.0+-0.0  tasks=0.00
  [Flow  260]  loss=0.0187  R=0.0+-0.0  tasks=0.00
  [Flow  280]  loss=0.0172  R=0.0+-0.0  tasks=0.00
  [Flow  300]  loss=0.0170  R=0.0+-0.0  tas